# XGBoost Model Training for Crypto Price Direction Prediction

This notebook trains an XGBoost classifier to predict crypto price direction (UP/DOWN/NEUTRAL) using technical indicators and features.

## Overview
- **Task**: Multi-class classification (3 classes: DOWN, NEUTRAL, UP)
- **Model**: XGBoost with hyperparameter tuning via Optuna
- **Data Split**: Time-series split (no shuffling to prevent look-ahead bias)
- **Evaluation**: F1-score (weighted), accuracy, classification report

## Google Colab Setup (Optional)

If you're running this on **Google Colab** with GPU:

1. **Enable GPU Runtime**:
   - Go to `Runtime` → `Change runtime type`
   - Select `GPU` (T4, V100, or A100) under Hardware accelerator
   - Click `Save`

2. **Verify GPU is available**:
   ```python
   !nvidia-smi
   ```

3. **Install dependencies** (if not already installed):
   ```python
   !pip install xgboost optuna pandas numpy scikit-learn ta joblib
   ```

4. **Mount Google Drive** (optional - to save models):
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   OUTPUT_DIR = Path('/content/drive/MyDrive/models')
   ```

The notebook will **automatically detect and use GPU** if available!

In [ ]:
!pip install -r requirements.txt

## 1. Setup and Imports

In [ ]:
import json
from pathlib import Path
import os

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import optuna
import joblib

# Custom modules
from build_features import FEATURE_COLUMNS, prepare_dataset

# Display settings
pd.set_option('display.max_columns', None)

print("✓ All packages imported successfully")

## 2. Configuration

Set your training parameters here:

In [ ]:
# Data parameters
DATA_DIR = Path("../data_5m")  # 5-minute OHLCV data
SYMBOLS = ["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT"]  # All major coins
THRESHOLD = 0.002  # Return threshold for UP/DOWN labels (0.2% for 5m candles)
TIMEFRAME = "5m"  # Base timeframe (5m recommended)

# Training parameters
TRAIN_RATIO = 0.8  # 80% train, 20% validation
N_TRIALS = 50  # Number of Optuna hyperparameter optimization trials

# Output
OUTPUT_DIR = Path("models")

# GPU Configuration - Automatically detect and use GPU if available
USE_GPU = False
TREE_METHOD = "hist"  # Default to CPU

try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode == 0:
        USE_GPU = True
        TREE_METHOD = "gpu_hist"
        print("✓ GPU detected! Using GPU-accelerated training")
        print(result.stdout.decode()[:500])  # Show first 500 chars of nvidia-smi
except:
    print("ℹ No GPU detected. Using CPU training")

print(f"\nTraining configuration:")
print(f"  Device: {'GPU (CUDA)' if USE_GPU else 'CPU'}")
print(f"  Tree method: {TREE_METHOD}")
print(f"  Symbols: {SYMBOLS}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Return threshold: {THRESHOLD*100:.3f}%")
print(f"  Train/Val split: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}")
print(f"  Optuna trials: {N_TRIALS}")
print(f"  Output directory: {OUTPUT_DIR}")

## 3. Data Loading and Preparation

In [ ]:
# List available data files in DATA_DIR
print(f"Checking data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}\n")

if DATA_DIR.exists():
    files = sorted(DATA_DIR.glob("*.parquet"))
    if files:
        print(f"Found {len(files)} parquet file(s):\n")
        for f in files:
            file_size = os.path.getsize(f) / (1024 * 1024)  # Convert to MB
            print(f"  • {f.name:40s} ({file_size:.2f} MB)")
    else:
        print("⚠ No parquet files found in data directory")
        print("Run fetch_data.py to download historical data first")
else:
    print(f"⚠ Data directory does not exist: {DATA_DIR}")
    print("Create it and run fetch_data.py to download historical data")

In [ ]:
print("Loading and preparing dataset...")
X, y = prepare_dataset(DATA_DIR, SYMBOLS, THRESHOLD, TIMEFRAME)

print(f"\nDataset shape: {X.shape}")
print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"\nClass distribution:")
print(y.value_counts().sort_index())
print(f"\nClass distribution (%):")
print((y.value_counts(normalize=True) * 100).sort_index())

## 4. Train/Validation Split (Time-Series)

We use time-series split to avoid look-ahead bias - train on earlier data, validate on later data.

In [ ]:
def time_series_split(X: pd.DataFrame, y: pd.Series, train_ratio: float = 0.8):
    """Split data by time, no shuffling."""
    split_idx = int(len(X) * train_ratio)
    
    X_train = X.iloc[:split_idx]
    y_train = y.iloc[:split_idx]
    X_val = X.iloc[split_idx:]
    y_val = y.iloc[split_idx:]
    
    return X_train, X_val, y_train, y_val


X_train, X_val, y_train, y_val = time_series_split(X, y, TRAIN_RATIO)

print(f"Train set: {len(X_train):,} samples")
print(f"Val set:   {len(X_val):,} samples")
print(f"\nTrain class distribution:")
print(y_train.value_counts().sort_index())
print(f"\nVal class distribution:")
print(y_val.value_counts().sort_index())

## 5. Hyperparameter Optimization with Optuna

We use Optuna to find the best hyperparameters for XGBoost, optimizing for weighted F1-score.

In [ ]:
def objective(trial, X_train, y_train, X_val, y_val):
    """Optuna objective for hyperparameter tuning."""
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": TREE_METHOD,  # Use GPU if available
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
    }
    
    # Add GPU-specific parameters
    if USE_GPU:
        params["device"] = "cuda"
    
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    
    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred, average="weighted")
    
    return f1

In [ ]:
print(f"Running Optuna optimization with {N_TRIALS} trials...")
print("This may take several minutes...\n")

study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: objective(trial, X_train, y_train, X_val, y_val),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(f"\n{'='*60}")
print(f"Best trial F1-score: {study.best_trial.value:.4f}")
print(f"{'='*60}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key:20s}: {value}")

## 6. Train Final Model with Best Hyperparameters

In [ ]:
best_params = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": TREE_METHOD,  # Use GPU if available
    "random_state": 42,
    **study.best_trial.params,
}

# Add GPU-specific parameters
if USE_GPU:
    best_params["device"] = "cuda"

print("Training final model with best hyperparameters...")
print(f"Using: {TREE_METHOD} ({'GPU' if USE_GPU else 'CPU'})\n")

model = xgb.XGBClassifier(**best_params)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=True,
)

print("\nTraining complete!")

## 7. Model Evaluation and Validation

In [ ]:
# Predictions
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)

# Calculate metrics
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
train_f1 = f1_score(y_train, y_train_pred, average="weighted")
val_f1 = f1_score(y_val, y_val_pred, average="weighted")

print("="*60)
print("MODEL EVALUATION RESULTS")
print("="*60)
print(f"\nTrain Accuracy:      {train_acc:.4f}")
print(f"Val Accuracy:        {val_acc:.4f}")
print(f"\nTrain F1 (weighted): {train_f1:.4f}")
print(f"Val F1 (weighted):   {val_f1:.4f}")
print(f"\nOverfitting check:   {(train_acc - val_acc):.4f} (< 0.05 is good)")
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Validation Set)")
print("="*60)
print(classification_report(y_val, y_val_pred, target_names=["DOWN", "NEUTRAL", "UP"]))

### Confusion Matrix (Text)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_val, y_val_pred)
cm_norm = confusion_matrix(y_val, y_val_pred, normalize='true')

print("Confusion Matrix (Counts):")
print("="*60)
print(f"{'':>12} {'Pred DOWN':>12} {'Pred NEUTRAL':>15} {'Pred UP':>12}")
print("="*60)
print(f"{'True DOWN':>12} {cm[0,0]:>12,} {cm[0,1]:>15,} {cm[0,2]:>12,}")
print(f"{'True NEUTRAL':>12} {cm[1,0]:>12,} {cm[1,1]:>15,} {cm[1,2]:>12,}")
print(f"{'True UP':>12} {cm[2,0]:>12,} {cm[2,1]:>15,} {cm[2,2]:>12,}")
print("="*60)

print("\nConfusion Matrix (Normalized):")
print("="*60)
print(f"{'':>12} {'Pred DOWN':>12} {'Pred NEUTRAL':>15} {'Pred UP':>12}")
print("="*60)
print(f"{'True DOWN':>12} {cm_norm[0,0]:>11.2%} {cm_norm[0,1]:>14.2%} {cm_norm[0,2]:>11.2%}")
print(f"{'True NEUTRAL':>12} {cm_norm[1,0]:>11.2%} {cm_norm[1,1]:>14.2%} {cm_norm[1,2]:>11.2%}")
print(f"{'True UP':>12} {cm_norm[2,0]:>11.2%} {cm_norm[2,1]:>14.2%} {cm_norm[2,2]:>11.2%}")
print("="*60)

### Feature Importance (Top 20)

In [ ]:
# Get feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print("="*60)
print(f"{'Rank':<6} {'Feature':<25} {'Importance':<15}")
print("="*60)
for i, row in enumerate(importance_df.head(20).itertuples(), 1):
    print(f"{i:<6} {row.feature:<25} {row.importance:<15.6f}")
print("="*60)

### Prediction Confidence Statistics

In [ ]:
# Get max probability for each prediction (confidence)
max_proba = y_val_proba.max(axis=1)

print("Prediction Confidence Statistics:")
print("="*60)
print(f"  Mean confidence:   {max_proba.mean():.4f}")
print(f"  Median confidence: {np.median(max_proba):.4f}")
print(f"  Min confidence:    {max_proba.min():.4f}")
print(f"  Max confidence:    {max_proba.max():.4f}")
print(f"  Std deviation:     {max_proba.std():.4f}")
print("="*60)

## 8. Save Model and Metrics

In [ ]:
def save_model(model: xgb.XGBClassifier, output_dir: Path, metrics: dict):
    """Save model in multiple formats."""
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save XGBoost native format
    model_path = output_dir / "xgboost_model.json"
    model.save_model(model_path)
    print(f"✓ Saved XGBoost model to {model_path}")
    
    # Save joblib format
    joblib_path = output_dir / "xgboost_model.joblib"
    joblib.dump(model, joblib_path)
    print(f"✓ Saved joblib model to {joblib_path}")
    
    # Save metrics
    metrics_path = output_dir / "metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"✓ Saved metrics to {metrics_path}")
    
    # Save feature names
    features_path = output_dir / "features.json"
    with open(features_path, "w") as f:
        json.dump(FEATURE_COLUMNS, f, indent=2)
    print(f"✓ Saved feature names to {features_path}")


# Prepare metrics dictionary
metrics = {
    "train_accuracy": float(train_acc),
    "val_accuracy": float(val_acc),
    "train_f1_weighted": float(train_f1),
    "val_f1_weighted": float(val_f1),
    "train_size": len(X_train),
    "val_size": len(X_val),
    "symbols": SYMBOLS,
    "threshold": THRESHOLD,
    "best_params": best_params,
    "n_features": len(FEATURE_COLUMNS),
}

# Save everything
print("\nSaving model and artifacts...")
save_model(model, OUTPUT_DIR, metrics)
print("\n✓ All artifacts saved successfully!")

## 9. Final Training Report

In [ ]:
print("\n" + "="*70)
print("FINAL TRAINING REPORT")
print("="*70)

print(f"\n📊 TRAINING DATA")
print(f"  Symbols: {', '.join(SYMBOLS)}")
print(f"  Total samples: {len(X):,}")
print(f"  Features: {len(FEATURE_COLUMNS)}")
print(f"  Return threshold: {THRESHOLD*100:.3f}%")

print(f"\n🔀 DATA SPLIT")
print(f"  Train: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val:   {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)")

print(f"\n⚙️  HYPERPARAMETERS (Best Trial)")
for key, value in study.best_trial.params.items():
    print(f"  {key:20s}: {value}")

print(f"\n📈 PERFORMANCE METRICS")
print(f"  Train Accuracy:  {train_acc:.4f}")
print(f"  Val Accuracy:    {val_acc:.4f}")
print(f"  Train F1:        {train_f1:.4f}")
print(f"  Val F1:          {val_f1:.4f}")
print(f"  Overfitting gap: {(train_acc - val_acc):.4f}")

print(f"\n✅ VALIDATION STATUS")
overfitting_ok = (train_acc - val_acc) < 0.05
f1_ok = val_f1 > 0.40  # Reasonable threshold for multi-class
print(f"  Overfitting check:  {'✓ PASS' if overfitting_ok else '⚠ CHECK REQUIRED'} (gap < 0.05)")
print(f"  F1-score check:     {'✓ PASS' if f1_ok else '⚠ CHECK REQUIRED'} (F1 > 0.40)")
print(f"  Overall status:     {'✓ READY FOR EXPORT' if (overfitting_ok and f1_ok) else '⚠ REVIEW RECOMMENDED'}")

print(f"\n💾 OUTPUT FILES")
print(f"  Directory: {OUTPUT_DIR}")
print(f"  1. xgboost_model.json    (XGBoost native format)")
print(f"  2. xgboost_model.joblib  (Joblib format)")
print(f"  3. features.json         (Feature names)")
print(f"  4. metrics.json          (Performance metrics)")

print("\n" + "="*70)
print("✓ TRAINING COMPLETED SUCCESSFULLY")
print("="*70)
print("\nNext step: Run export_model.ipynb to convert to ONNX format for Go")

## Next Steps

1. **Export to ONNX** (for Go inference): Run `export_model.ipynb`
2. **Backtest**: Use the saved model in the Go backtesting engine
3. **Paper Trading**: Test with live data before real deployment
4. **Retrain**: Periodically retrain with fresh data to adapt to market changes

## Notes

- Model is saved in multiple formats for flexibility
- Use time-series cross-validation for more robust evaluation
- Consider feature selection to reduce model complexity
- Monitor model performance degradation in production
- Retrain when validation metrics drop significantly